In [2]:
import ewatercycle.forcing
import ewatercycle.observation.grdc
import ewatercycle.analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from pathlib import Path
from cartopy.io import shapereader
from rich import print
import matplotlib.pyplot as plt
import xarray as xr


/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [ ]:
file = "Interception.xlsx"
catchment = pd.read_excel(file, index_col= 0, parse_dates= True, skiprows = 2)
catchment.columns = ["precip1", "ep1", "precip2", "ep2"]
catchment = catchment.dropna()
catchment.head()



C_evap.name = "EP mm/d" 
C_prec.name = "P mm/d"
C_obs_data = C_obs['Q'].to_xarray().rename({'Day': 'time'})
C_obs_data.name = "Q mm/d"

df = xr.merge([C_prec, C_evap, C_obs_data]).to_dataframe()
data = df[["P mm/d", "EP mm/d", "Q mm/d"]].dropna()
data.index = pd.to_datetime(data.index)
c_pre_deforestation = data.loc["1991-01-01":"1994-12-31"]
c_post_deforestation = data.loc["1996-01-01":"1999-12-31"]
c_pre_deforestation

In [ ]:
si_max = 2.5  # mm
Si1 = 0
Si2 = 0

catchment["Pe_ger"] = 0.0
catchment["Pe_scot"] = 0.0

for i in range(len(catchment)):
    # Germany
    P = catchment["precip1"].iloc[i]
    Ep = catchment["ep1"].iloc[i]

    Si_temp = Si1 + P

    if Si_temp > si_max:
        Pe = Si_temp - si_max
        Si_temp = si_max
    else:
        Pe = 0

    Ei = min(Ep, Si_temp)
    Si1 = Si_temp - Ei

    catchment.loc[catchment.index[i], "Pe_ger"] = Pe

    # Scotland
    P = catchment["precip2"].iloc[i]
    Ep = catchment["ep2"].iloc[i]

    Si_temp = Si2 + P

    if Si_temp > si_max:
        Pe = Si_temp - si_max
        Si_temp = si_max
    else:
        Pe = 0

    Ei = min(Ep, Si_temp)
    Si2 = Si_temp - Ei

    catchment.loc[catchment.index[i], "Pe_scot"] = Pe

In [ ]:
ratio_ger = catchment["Pe_ger"].mean() / catchment["precip1"].mean()

print(f"Germany Pe/P = {ratio_ger:.2f}")

catchment[["precip1", "Pe_ger"]].plot()
plt.ylabel("mm/d")
plt.title("Germany precipitation vs throughfall")
plt.show()


In [1]:
plt.scatter(catchment["precip1"], catchment["Pe_ger"], label="Germany")

plt.xlabel("Precipitation (mm/d)")
plt.ylabel("Throughfall Pe (mm/d)")
plt.legend()

NameError: name 'plt' is not defined

In [ ]:
file2 = "Rootzone Storage Capacity.xlsx"
rootzone = pd.read_excel(file2, index_col= 0, parse_dates= True, skiprows = 2)
rootzone = rootzone.dropna()

rootzone.columns = ["P_pre", "EP_pre", "Q_pre", "Date_post", "P_post", "EP_post", "Q_post"]

rootzone.head()

In [ ]:
# pre-deforestation
pre = rootzone[["P_pre", "EP_pre", "Q_pre"]].copy()

# post-deforestation
post = rootzone[["Date_post", "P_post", "EP_post", "Q_post"]].copy()
post = post.dropna()
post["Date_post"] = pd.to_datetime(post["Date_post"])
post = post.set_index("Date_post")
print(pre.head())
print(post.head())

pre["ET"] = pre["P_pre"] - pre["Q_pre"]
post["ET"] = post["P_post"] - post["Q_post"]
pre["ET"] = pre["ET"].clip(lower=0)
post["ET"] = post["ET"].clip(lower=0)

In [ ]:

# 1) Long-term mean ET from the water balance
ET_mean_pre = pre["P_pre"].mean() - pre["Q_pre"].mean()
ET_mean_post = post["P_post"].mean() - post["Q_post"].mean()

# 2) Scale daily ET with daily EP
pre["ET_daily"] = (pre["EP_pre"] / pre["EP_pre"].mean()) * ET_mean_pre
post["ET_daily"] = (post["EP_post"] / post["EP_post"].mean()) * ET_mean_post

# 3) Compute storage deficit for pre-deforestation period
sd_pre = [0.0]

for i in range(1, len(pre)):
    new_sd = min(0, sd_pre[i-1] + pre["P_pre"].iloc[i] - pre["ET_daily"].iloc[i])
    sd_pre.append(new_sd)

pre["SD"] = sd_pre

# 4) Compute storage deficit for post-deforestation period
sd_post = [0.0]

for i in range(1, len(post)):
    new_sd = min(0, sd_post[i-1] + post["P_post"].iloc[i] - post["ET_daily"].iloc[i])
    sd_post.append(new_sd)

post["SD"] = sd_post

# 5) Root-zone storage capacity = maximum storage deficit
SR_pre = (-pre["SD"]).max()
SR_post = (-post["SD"]).max()

print("SR pre-deforestation =", SR_pre, "mm")
print("SR post-deforestation =", SR_post, "mm")

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(pre["SD"], label="Pre-deforestation")
plt.plot(post["SD"], label="Post-deforestation")

plt.axhline(0)
plt.ylabel("Storage deficit (mm)")
plt.title("Storage deficit through time")
plt.legend()


x_pre = pre["EP_pre"].mean() / pre["P_pre"].mean()
y_pre = ET_mean_pre / pre["P_pre"].mean()

x_post = post["EP_post"].mean() / post["P_post"].mean()
y_post = ET_mean_post / post["P_post"].mean()

plt.figure(figsize=(6,6))
plt.scatter(x_pre, y_pre, label="Pre-deforestation")
plt.scatter(x_post, y_post, label="Post-deforestation")

plt.xlabel("EP / P")
plt.ylabel("ET / P")
plt.title("Budyko space")
plt.legend()


plt.figure(figsize=(10,4))
plt.plot(pre["P_pre"], label="Precipitation")
plt.plot(pre["ET_daily"], label="ET")

plt.ylabel("mm/day")
plt.title("Daily precipitation vs ET")
plt.legend()